# 02 — Data preprocessing for BM25
Find the project root by walking upward, using notebook 01's approach. Import the shared `Analyzer`: a class that turns text into tokens (searchable pieces). Both documents and future queries must use this class.

In [1]:
import json
import sys
from collections import Counter
from importlib.metadata import version
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find the nearest ancestor containing the data-pipeline directory."""
    for directory in (start, *start.parents):
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Run this notebook inside the legal-rag project')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
BM25_DIR = PROJECT_ROOT / 'data-pipeline' / 'notebook_bm25'
INPUT_PATH = DATA_DIR / 'processed' / 'articles.jsonl'
NORMALIZED_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
OUTPUT_DIR = DATA_DIR / 'bm25' / 'preprocessed'
sys.path.insert(0, str(BM25_DIR))
from bm25_index import InvertedIndex, canonical_id
from text_pipeline import (Analyzer, context_text, normalization_report,
                           read_jsonl, selected_record_indices, token_statistics)

print('Project root:', PROJECT_ROOT)
print('Input:', INPUT_PATH)
print('Output:', OUTPUT_DIR)

Project root: /home/panha/ir/project/legal-rag
Input: /home/panha/ir/project/legal-rag/data-pipeline/data/processed/articles.jsonl
Output: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/preprocessed


Load every raw article record. Context is optional extra text taken from the document title, article number, chapter title, and section title, matching the dense track. Missing fields are skipped; a literal `unknown` title is safe. `INCLUDE_CONTEXT=False` keeps this run directly comparable with notebook 01.

In [2]:
articles = read_jsonl(INPUT_PATH)
assert articles, 'The article input is empty'
INCLUDE_CONTEXT = False  # Turn on only when you want document context included.
TOKENIZER = 'khmercut'  # Alternatives: 'khmernltk' and 'syllable_ngrams'.
SYLLABLE_NGRAM_MIN = 1  # The alternative includes individual orthographic syllables.
SYLLABLE_NGRAM_MAX = 2  # It also includes pairs of adjacent syllables.
STOPWORD_PATH = BM25_DIR / 'stopwords_km.txt'
analyzer = Analyzer(tokenizer=TOKENIZER, include_context=INCLUDE_CONTEXT,
                    stopword_path=STOPWORD_PATH,
                    syllable_ngram_min=SYLLABLE_NGRAM_MIN,
                    syllable_ngram_max=SYLLABLE_NGRAM_MAX)
print('Loaded records:', len(articles))
print('Available fields:', sorted(set().union(*(article.keys() for article in articles))))
print('Context fields: document_title, article_number, chapter_title, section_title')
print('Sample context:', context_text(articles[0]))
print('INCLUDE_CONTEXT:', INCLUDE_CONTEXT, '| tokenizer:', TOKENIZER)
print('Stopwords:', sorted(analyzer.stopwords))
print('Kept legal word ត្រូវ:', 'ត្រូវ' not in analyzer.stopwords)

Loaded records: 20268


Available fields: ['article_id', 'article_number', 'chapter_number', 'chapter_title', 'document_id', 'document_number', 'document_title', 'document_type', 'id', 'pdf_url', 'section_number', 'section_title', 'source_url', 'text']
Context fields: document_title, article_number, chapter_title, section_title
Sample context: 1.txt | មាត្រា 1
INCLUDE_CONTEXT: False | tokenizer: khmercut
Stopwords: ['ចំពោះ', 'ដូចម្តេច', 'ដែល', 'ដោយ', 'ណា', 'តើ', 'និង', 'នៃ', 'នៅ', 'ពី', 'ហើយ', 'ហេតុអ្វី', 'អ្វី']
Kept legal word ត្រូវ: True


Check both installed word segmenters on a Khmer sentence before processing the collection. `khmer-nltk` is the package name; Python imports it as `khmernltk`. The third option builds overlapping groups of one or two rule-based Khmer orthographic syllables (written units, not a pronunciation model). No backend silently falls back to another.

In [3]:
SEGMENTATION_SAMPLE = 'ប្រជាពលរដ្ឋត្រូវគោរពច្បាប់។ មាត្រា១២ ស្តីពីសិទ្ធិ និងកាតព្វកិច្ច។'
print('Sample:', SEGMENTATION_SAMPLE)
print('Installed versions:', {name: version(name) for name in ['khmercut', 'khmer-nltk', 'regex']})
for tokenizer in ['khmercut', 'khmernltk', 'syllable_ngrams']:
    trial = Analyzer(tokenizer=tokenizer, stopword_path=STOPWORD_PATH,
                     syllable_ngram_min=SYLLABLE_NGRAM_MIN,
                     syllable_ngram_max=SYLLABLE_NGRAM_MAX)
    tokens = trial.analyze(SEGMENTATION_SAMPLE)
    assert tokens, f'{tokenizer} produced no sample tokens'
    print(tokenizer, tokens)
print('All three tokenizer choices ran successfully.')

Sample: ប្រជាពលរដ្ឋត្រូវគោរពច្បាប់។ មាត្រា១២ ស្តីពីសិទ្ធិ និងកាតព្វកិច្ច។
Installed versions: {'khmercut': '0.2.0', 'khmer-nltk': '1.6', 'regex': '2026.11.7'}
khmercut ['ប្រជាពលរដ្ឋ', 'ត្រូវ', 'គោរព', 'ច្បាប់', 'មាត្រា', '12', 'ស្តីពី', 'សិទ្ធិ', 'កាតព្វកិច្ច']


| 2026-11-07 23:22:02,158 | INFO | khmer-nltk | Loaded model from /home/panha/ir/project/legal-rag/.venv/lib/python3.14/site-packages/khmernltk/word_tokenize/sklearn_crf_ner_10000.sav |


khmernltk ['ប្រជាពលរដ្ឋ', 'ត្រូវ', 'គោរព', 'ច្បាប់', 'មាត្រា', '12', 'ស្តីពី', 'សិទ្ធិ', 'កាតព្វកិច្ច']
syllable_ngrams ['ប្រ', 'ជា', 'ព', 'ល', 'រ', 'ដ្ឋ', 'ត្រូ', 'វ', 'គោ', 'រ', 'ព', 'ច្បា', 'ប់', 'ប្រជា', 'ជាព', 'ពល', 'លរ', 'រដ្ឋ', 'ដ្ឋត្រូ', 'ត្រូវ', 'វគោ', 'គោរ', 'រព', 'ពច្បា', 'ច្បាប់', 'មា', 'ត្រា', 'មាត្រា', '12', 'ស្តី', 'សិ', 'ទ្ធិ', 'ស្តីពី', 'ពីសិ', 'សិទ្ធិ', 'និ', 'ង', 'កា', 'ត', 'ព្វ', 'កិ', 'ច្ច', 'ងកា', 'កាត', 'តព្វ', 'ព្វកិ', 'កិច្ច']
All three tokenizer choices ran successfully.


Check normalization before folding digits or adding context. The normalizer and its helper/constants are copied unchanged from notebook 05. That notebook writes normalized text in `search_text`, while leaving `text` raw. If its output exists, compare every record in source order, including repeated IDs, and stop on any mismatch. If it is absent, report that the comparison could not be made.

In [4]:
normalization = normalization_report(articles, NORMALIZED_PATH)
if normalization['available']:
    print(f"Normalization matches: {normalization['matches']:,} / {normalization['checked']:,}")
    print('Mismatches:', normalization['mismatches'])
    assert normalization['mismatches'] == 0, 'Normalization differs from notebook 05 output'
else:
    print('Normalization reference is absent:', NORMALIZED_PATH)
    print('Normalization reference matches: 0 checked; comparison unavailable.')

Normalization reference is absent: /home/panha/ir/project/legal-rag/data-pipeline/data/processed/articles_normalized.jsonl
Normalization reference matches: 0 checked; comparison unavailable.


Trace one real article through every step. Khmer digits become ASCII digits, and digit runs are separated from neighboring letters. Lowercasing changes Latin text; it can leave this sample unchanged. Punctuation and symbols become spaces to avoid joining words. Stopwords are removed only after segmentation. Legal words such as `ត្រូវ`, negation, and prohibitions are kept.

In [5]:
sample_article = articles[0]
print('Canonical article ID:', canonical_id(sample_article))
print('Raw text:', sample_article['text'])
sample_text = analyzer.document_text(sample_article)
print('Text with context policy applied:', sample_text)
for name, result in analyzer.explain(sample_text).items():
    print('\n' + name + ':')
    print(result)
assert analyzer.explain(sample_text)['6. Stopwords removed'] == analyzer.analyze_document(sample_article)
print('Trace tokens match normal document analysis.')

Canonical article ID: 1-1
Raw text: មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ៤៨ មាត្រា ៧៥ មាត្រា ៧៧ មាត្រា ៧៨ និងមាត្រា ៨០ នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម/០១១៥/០០១ ចុះថ្ងៃទី ៩ ខែ មករា ឆ្នាំ២០១៥ ត្រូវបានធ្វើវិសោធនកម្មដូចតទៅ ៖
Text with context policy applied: មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ៤៨ មាត្រា ៧៥ មាត្រា ៧៧ មាត្រា ៧៨ និងមាត្រា ៨០ នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម/០១១៥/០០១ ចុះថ្ងៃទី ៩ ខែ មករា ឆ្នាំ២០១៥ ត្រូវបានធ្វើវិសោធនកម្មដូចតទៅ ៖

1. Normalized:
មាត្រា ៨ មាត្រា ៤០ មាត្រា ៤១ មាត្រា ៤៧ មាត្រា ៤៨ មាត្រា ៧៥ មាត្រា ៧៧ មាត្រា ៧៨ និងមាត្រា ៨០ នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម/០១១៥/០០១ ចុះថ្ងៃទី ៩ ខែ មករា ឆ្នាំ២០១៥ ត្រូវបានធ្វើវិសោធនកម្មដូចតទៅ ៖

2. Digits folded and separated:
មាត្រា 8 មាត្រា 40 មាត្រា 41 មាត្រា 47 មាត្រា 48 មាត្រា 75 មាត្រា 77 មាត្រា 78 និងមាត្រា 80 នៃច្បាប់ស្តីពីចរាចរណ៍ផ្លូវគោក ដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រមលេខ នស/រកម

Analyze every input record with the same steps used for queries. Keep all records in the output, including repeated article IDs, as requested. For comparison with notebook 01, choose the longest raw text per canonical ID and keep the first record on ties. Count segmented tokens before stopword removal on that same selected collection so you can review candidates fairly.

In [6]:
selected_positions = selected_record_indices(articles)
selected_set = set(selected_positions)
PROGRESS_INTERVAL = 2000  # Report progress after each 2,000 records.
segmented_counts = Counter()
token_records = []
for position, article in enumerate(articles):
    prepared = analyzer.prepare(analyzer.document_text(article))
    segmented = analyzer.segment(prepared)
    tokens = analyzer.remove_stopwords(segmented)
    token_records.append({'article_id': canonical_id(article),
                          'source_record_index': position, 'tokens': tokens})
    if position in selected_set:
        segmented_counts.update(segmented)
    if (position + 1) % PROGRESS_INTERVAL == 0:
        print('Analyzed records:', position + 1, flush=True)
assert len(token_records) == len(articles)
print('Analyzed all records:', len(token_records))
print('Unique articles for comparison:', len(selected_positions))
print('Repeated-ID records retained in output:', len(articles) - len(selected_positions))
print('Sample token record:', token_records[0])

Analyzed records: 2000


Analyzed records: 4000


Analyzed records: 6000


Analyzed records: 8000


Analyzed records: 10000


Analyzed records: 12000


Analyzed records: 14000


Analyzed records: 16000


Analyzed records: 18000


Analyzed records: 20000


Analyzed all records: 20268
Unique articles for comparison: 17540
Repeated-ID records retained in output: 2728
Sample token record: {'article_id': '1-1', 'source_record_index': 0, 'tokens': ['មាត្រា', '8', 'មាត្រា', '40', 'មាត្រា', '41', 'មាត្រា', '47', 'មាត្រា', '48', 'មាត្រា', '75', 'មាត្រា', '77', 'មាត្រា', '78', 'មាត្រា', '80', 'ច្បាប់', 'ស្តីពី', 'ចរាចរណ៍', 'ផ្លូវគោក', 'ត្រូវ', 'ប្រកាស', 'ឱ្យ', 'ប្រើ', 'ព្រះរាជ', 'ក្រម', 'លេខ', 'នស', 'រក', 'ម', '0115', '001', 'ចុះ', 'ថ្ងៃទី', '9', 'ខែ', 'មករា', 'ឆ្នាំ', '2015', 'ត្រូវបាន', 'ធ្វើ', 'វិសោធនកម្ម', 'ដូចតទៅ']}


Review the 50 most frequent tokens after segmentation and before stopword removal. Frequency means total occurrences in the selected articles. `[REMOVE]` marks entries in the editable stopword file; `[KEEP]` marks all others. English words remain unless explicitly listed: this short stopword list targets Khmer grammatical words.

In [7]:
STOPWORD_CANDIDATE_COUNT = 50  # Display enough common tokens for manual review.
ranked_candidates = sorted(segmented_counts.items(), key=lambda item: (-item[1], item[0]))
print('Stopword candidates: token | occurrences | decision')
for token, count in ranked_candidates[:STOPWORD_CANDIDATE_COUNT]:
    decision = '[REMOVE]' if token in analyzer.stopwords else '[KEEP]'
    print(repr(token), count, decision)
print('Stopword file:', STOPWORD_PATH)
print(STOPWORD_PATH.read_text(encoding='utf-8'))

Stopword candidates: token | occurrences | decision
'ដែល' 38313 [REMOVE]
'និង' 35708 [REMOVE]
'ត្រូវ' 28434 [KEEP]
'មាន' 28130 [KEEP]
'នៃ' 27634 [REMOVE]
'ឬ' 27324 [KEEP]
'ក្នុង' 21078 [KEEP]
'បាន' 18759 [KEEP]
'ដោយ' 18358 [REMOVE]
'របស់' 17434 [KEEP]
'នេះ' 16966 [KEEP]
'ជា' 16223 [KEEP]
'តាម' 13464 [KEEP]
'ច្បាប់' 12521 [KEEP]
'the' 11078 [KEEP]
'ពី' 10656 [REMOVE]
'មិន' 10262 [KEEP]
'នោះ' 9441 [KEEP]
'ឆ្នាំ' 8953 [KEEP]
'អាច' 8821 [KEEP]
'of' 8793 [KEEP]
'1' 8193 [KEEP]
'ទៅ' 8167 [KEEP]
'នៅក្នុង' 8151 [KEEP]
'មាត្រា' 7755 [KEEP]
'ឱ្យ' 7591 [KEEP]
'ធ្វើ' 7297 [KEEP]
'មួយ' 7185 [KEEP]
'លើ' 7109 [KEEP]
'សិទ្ធិ' 6992 [KEEP]
'2' 6840 [KEEP]
'កំណត់' 6776 [KEEP]
'ទី' 6706 [KEEP]
'ខ្លួន' 6465 [KEEP]
'ដល់' 6092 [KEEP]
'នៅ' 5852 [REMOVE]
'ដើម្បី' 5616 [KEEP]
'ចំពោះ' 5580 [REMOVE]
'រដ្ឋ' 5563 [KEEP]
'000' 5369 [KEEP]
'នឹង' 5306 [KEEP]
'ជាតិ' 5291 [KEEP]
'ត្រូវបាន' 5289 [KEEP]
'ចែង' 5263 [KEEP]
'ទៀត' 5243 [KEEP]
'ៗ' 5234 [KEEP]
'សម្រាប់' 5215 [KEEP]
'សមាជិក' 5121 [KEEP]
'to' 5080 [KEEP]
'ប្រាក់'

Compare the same unique articles before and after preprocessing. Vocabulary is the number of distinct tokens. The one-time share counts distinct tokens occurring exactly once in the collection, divided by vocabulary size. Mean length is the average token count per article. Longer token counts after segmentation are expected because Khmer phrases have been split into words. Also check the raw numbers against notebook 01's saved index when available.

In [8]:
raw_documents = [articles[position]['text'].split() for position in selected_positions]
clean_documents = [token_records[position]['tokens'] for position in selected_positions]
before = token_statistics(raw_documents)
after = token_statistics(clean_documents)
print(f"{'Metric':<30} {'Raw':>16} {'Preprocessed':>16}")
for label, key in [('Documents', 'documents'), ('Vocabulary', 'vocabulary')]:
    print(f"{label:<30} {before[key]:>16,} {after[key]:>16,}")
print(f"{'One-time vocabulary share':<30} {before['one_time_share']:>16.2%} {after['one_time_share']:>16.2%}")
print(f"{'Average tokens per document':<30} {before['average_tokens']:>16.2f} {after['average_tokens']:>16.2f}")
raw_index_dir = DATA_DIR / 'bm25' / 'index_raw'
if (raw_index_dir / 'index.json').exists():
    raw_index = InvertedIndex.load(raw_index_dir)
    assert raw_index.total_documents == before['documents']
    assert raw_index.collection_frequencies == dict(before['ranked_tokens'])
    assert raw_index.average_document_length == before['average_tokens']
    assert set(raw_index.document_lengths) == {token_records[p]['article_id'] for p in selected_positions}
    print('Raw statistics match notebook 01 saved index.')
else:
    print('Notebook 01 index absent; raw baseline recomputed from the same input and selection rule.')

Metric                                      Raw     Preprocessed
Documents                                17,540           17,540
Vocabulary                              193,323           24,932
One-time vocabulary share                81.71%           40.50%
Average tokens per document               33.07            88.66


Raw statistics match notebook 01 saved index.


Show the 20 most frequent tokens on each side. Counts are total occurrences. This makes it easy to see whether punctuation disappeared and which common words remain; frequent legal words are intentionally retained.

In [9]:
TOP_TOKEN_COUNT = 20  # Compare the twenty most common tokens on each side.
print(f"{'Rank':<5} {'Raw token (count)':<45} Preprocessed token (count)")
for rank in range(TOP_TOKEN_COUNT):
    raw_entry = before['ranked_tokens'][rank] if rank < len(before['ranked_tokens']) else None
    clean_entry = after['ranked_tokens'][rank] if rank < len(after['ranked_tokens']) else None
    print(f'{rank + 1:<5} {str(raw_entry):<45} {clean_entry}')

Rank  Raw token (count)                             Preprocessed token (count)
1     ('៖', 17285)                                  ('ត្រូវ', 28434)
2     ('។', 17249)                                  ('មាន', 28130)
3     ('the', 9905)                                 ('ឬ', 27324)
4     ('of', 8608)                                  ('ក្នុង', 21078)
5     ('-', 7882)                                   ('បាន', 18759)
6     ('ឬ', 7501)                                   ('របស់', 17434)
7     ('៕', 5197)                                   ('នេះ', 16966)
8     ('to', 4954)                                  ('ជា', 16223)
9     ('និង', 4749)                                 ('តាម', 13464)
10    ('ឆ្នាំ', 4460)                               ('ច្បាប់', 12521)
11    ('|', 3766)                                   ('the', 11078)
12    ('and', 3678)                                 ('មិន', 10262)
13    ('or', 3615)                                  ('នោះ', 9441)
14    ('in', 2863)                            

Inspect remaining suspicious tokens without deleting them. Show the longest tokens and any token mixing letters with digits; these are review hints, not proof of an error. Report full lengths and shorten only the displayed text of very long examples.

In [10]:
import unicodedata

NOISE_EXAMPLE_COUNT = 5  # Keep each review list short.
TOKEN_PREVIEW_LENGTH = 160  # Display at most 160 code points per long token.
longest = sorted((token for token, count in after['ranked_tokens']),
                 key=lambda token: (-len(token), token))
print('Longest remaining tokens: length | preview')
for token in longest[:NOISE_EXAMPLE_COUNT]:
    print(len(token), repr(token[:TOKEN_PREVIEW_LENGTH]), '...' if len(token) > TOKEN_PREVIEW_LENGTH else '')
mixed = [token for token in longest if any(char.isdigit() for char in token)
         and any(unicodedata.category(char).startswith('L') for char in token)]
print('Tokens mixing letters and digits:', len(mixed))
print('Mixed examples:', mixed[:NOISE_EXAMPLE_COUNT])
print('Standalone punctuation/symbol tokens remaining:', sum(
    all(unicodedata.category(char)[0] in {'P', 'S'} for char in token) for token in longest))

Longest remaining tokens: length | preview
27 'evgeliépþdikardambegáalsium' 
26 'ការកាត់ផ្តាច់ការផ្គត់ផ្គង់' 
25 'ការធ្វើដំណើរការជំនុំជម្រះ' 
25 'ត្រួតពិនិត្យភាពត្រឹមត្រូវ' 
25 'សេចក្តីត្រូវការវិជ្ជាជីវៈ' 
Tokens mixing letters and digits: 0
Mixed examples: []
Standalone punctuation/symbol tokens remaining: 0


Save one token record per input record as UTF-8 JSON lines, plus the full analyzer settings. The source record index distinguishes repeated canonical IDs. Notebook 03 should use `Analyzer.from_config(CONFIG_PATH)` or call `assert_compatible(CONFIG_PATH)` on its query analyzer before search. The contract includes stopword contents, context policy, digit and punctuation rules, tokenizer settings, dependency versions, and a hash of the analyzer code.

In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOKENS_PATH = OUTPUT_DIR / 'articles_tokens.jsonl'
CONFIG_PATH = OUTPUT_DIR / 'preprocess_config.json'
with TOKENS_PATH.open('w', encoding='utf-8') as output:
    for record in token_records:
        output.write(json.dumps(record, ensure_ascii=False) + '\n')
analyzer.save_config(CONFIG_PATH)
reloaded_records = read_jsonl(TOKENS_PATH)
assert reloaded_records == token_records
query_analyzer = Analyzer.from_config(CONFIG_PATH, stopword_path=STOPWORD_PATH)
assert query_analyzer.analyze(sample_text) == analyzer.analyze(sample_text)
assert query_analyzer.analyze_document(sample_article) == token_records[0]['tokens']
print('Saved and reloaded token records:', len(reloaded_records))
print('Token file:', TOKENS_PATH)
print('Configuration:', CONFIG_PATH)
print(json.dumps(analyzer.config(), ensure_ascii=False, indent=2))

Saved and reloaded token records: 20268
Token file: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/preprocessed/articles_tokens.jsonl
Configuration: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/preprocessed/preprocess_config.json
{
  "schema_version": 1,
  "steps": [
    "normalize_khmer",
    "fold_digits",
    "lowercase",
    "strip_punctuation",
    "segment",
    "remove_stopwords"
  ],
  "normalization": {
    "enabled": true,
    "version": "unicode-khmer-cluster-order-spelling-v2",
    "zero_width": "​‌‍﻿",
    "spelling_variants": {
      "សំលៀកបំពាក់": "សម្លៀកបំពាក់",
      "ចំរៀង": "ចម្រៀង",
      "ជ័យជម្នះ": "ជ័យជំនះ"
    }
  },
  "digits": {
    "enabled": true,
    "source": "០១២៣៤៥៦៧៨៩",
    "target": "0123456789",
    "letter_boundary_pattern": "(?<=[\\p{L}\\p{M}])(?=[0-9])|(?<=[0-9])(?=[\\p{L}\\p{M}])"
  },
  "lowercase": {
    "enabled": true,
    "method": "str.lower"
  },
  "punctuation": {
    "enabled": true,
    "unicode_category_prefixes": 

Prove that an incompatible query analyzer is rejected. Flip only the context setting and check that the saved contract catches it. This test does not change the saved settings or the active analyzer.

In [12]:
incompatible = Analyzer(tokenizer=TOKENIZER, include_context=not INCLUDE_CONTEXT,
                        stopword_path=STOPWORD_PATH,
                        syllable_ngram_min=SYLLABLE_NGRAM_MIN,
                        syllable_ngram_max=SYLLABLE_NGRAM_MAX)
try:
    incompatible.assert_compatible(CONFIG_PATH)
except ValueError as error:
    print('Expected rejection:', error)
else:
    raise AssertionError('An incompatible analyzer was accepted')
print('Final validation passed: saved tokens, shared query analysis, and mismatch rejection.')

Expected rejection: Analyzer configuration mismatch: INCLUDE_CONTEXT
Final validation passed: saved tokens, shared query analysis, and mismatch rejection.
